In [1]:
import logging

from tiled.client.container import Container
from bluesky.plans import *
from bluesky.callbacks import best_effort
from bluesky.callbacks.tiled_writer import TiledWriter
from bluesky.run_engine import RunEngine
from tiled.client import simple
from statistics import mean
from blop.plans import optimize
from blop.protocols import EvaluationFunction, Optimizer, AcquisitionPlan, \
    OptimizationProblem
from ophyd.sim import det5, jittery_motor1, jittery_motor2, det4, motor1, motor2

# Suppress noisy logs from httpx 
logging.getLogger("httpx").setLevel(logging.WARNING)

[WARNING 03-26 15:07:28] ax.service.utils.with_db_settings_base: Ax currently requires a sqlalchemy version below 2.0. This will be addressed in a future release. Disabling SQL storage in Ax for now, if you would like to use SQL storage please install Ax with mysql extras via `pip install ax-platform[mysql]`.


In [2]:
tiled_client = simple()
tiled_writer = TiledWriter(tiled_client)
bec = best_effort.BestEffortCallback()
bec.disable_plots()

RE = RunEngine({})
RE.subscribe(bec)
RE.subscribe(tiled_writer)

2026-03-26 15:07:30.641 INFO: Subprocess stdout: 
2026-03-26 15:07:30.642 INFO: Subprocess stderr: Database sqlite+aiosqlite:////tmp/tmpfkla26cb/catalog.db is new. Creating tables.
Database initialized.

Tiled version 0.2.9.dev2+gb98e34b37
2026-03-26 15:07:31.176 INFO: Tiled version 0.2.9.dev2+gb98e34b37
2026-03-26 15:07:31.181 INFO: Context impl SQLiteImpl.
2026-03-26 15:07:31.182 INFO: Will assume non-transactional DDL.


http://127.0.0.1:33025/api/v1?api_key=469bff0bfc94e7f5


1

In [ ]:
class DetectorEvaluation(EvaluationFunction):
    def __init__(self, tiled_client: Container):
        self.tiled_client = tiled_client
    
    def __call__(self, uid: str, suggestions: list[dict]) -> list[dict]:
        outcomes = []
        run = self.tiled_client[uid]
        intensity = run["primary/det4"].read()
        x = run["primary/motor1"].read()
        y = run["primary/motor2"].read()
        suggestion_ids = run.metadata["start"]["blop_suggestion_ids"]
        
        for idx, sid in enumerate(suggestion_ids):
            outcome = {
                "_id": sid,
                "intensity": intensity[idx],
                "motor1": x[idx],
                "motor2": y[idx]
            }
            outcomes.append(outcome)
        return outcomes

In [4]:
class RandomOptimizer(Optimizer):

    outcome = set()
    dof = set()
    idx = 0

    def __init__(self, dof: list):
        self.dof = dof

    def suggest(self, num_points: int | None = None) -> list[dict]:
        self.idx += num_points
        if len(self.outcome) == 0:
            x = mean(self.dof["motor1"])
            y = mean(self.dof["motor2"])
        else:
            x = self.outcome[0]["motor1"] + 2
            y = self.outcome[0]["motor2"] + 1

        return [{
            "_id": self.idx,
            "motor1": x,
            "motor2": y
        }]

    def ingest(self, points: list[dict]) -> None:
        self.outcome = points

In [22]:
def custom_acquire(suggestions, movables, readables, *args, **kwargs):
    print(suggestions, movables, readables)
    md = {"blop_suggestion_ids": [suggestion.get("_id", None) for suggestion in suggestions]}
    m1 = suggestions[0]["motor1"]
    m2 = suggestions[0]["motor2"]
    return (yield from scan(readables, motor1, m1, m1-5, motor2, m2, m2-5, num=5, md=md))

In [6]:
dof = {
    "motor1": (-5, 5),
    "motor2": (-5, 5)
}

In [23]:
optimization_problem = OptimizationProblem(
    readables=[det4],
    movables=[motor1, motor2],
    optimizer=RandomOptimizer(dof),
    evaluation_function=DetectorEvaluation(tiled_client),
    acquisition_plan=custom_acquire
)

In [24]:
RE(optimize(optimization_problem, iterations=5, n_points=1))

2026-03-26 15:11:48.004 INFO: Executing plan <bluesky.utils.Plan object at 0x7a2310557820>
2026-03-26 15:11:48.005 INFO: Change state on <bluesky.run_engine.RunEngine object at 0x7a235d478910> from 'idle' -> 'running'


[{'_id': 1, 'motor1': 0, 'motor2': 0}] [SynAxis(prefix='', name='motor1', read_attrs=['readback', 'setpoint'], configuration_attrs=['velocity', 'acceleration']), SynAxis(prefix='', name='motor2', read_attrs=['readback', 'setpoint'], configuration_attrs=['velocity', 'acceleration'])] [Syn2DGauss(prefix='', name='det4', read_attrs=['val'], configuration_attrs=['Imax', 'center', 'sigma', 'noise', 'noise_multiplier'])]


Transient Scan ID: 13     Time: 2026-03-26 15:11:48
Persistent Unique Scan ID: '97a56dad-f73f-4f11-8533-3b6c02d99422'
New stream: 'primary'
+-----------+------------+------------+------------+------------+
|   seq_num |       time |     motor1 |     motor2 |       det4 |
+-----------+------------+------------+------------+------------+
|         1 | 15:11:48.0 |      0.000 |      0.000 |      1.000 |
|         2 | 15:11:48.0 |     -1.250 |     -1.250 |      0.210 |
|         3 | 15:11:48.0 |     -2.500 |     -2.500 |      0.002 |
|         4 | 15:11:48.1 |     -3.750 |    

2026-03-26 15:11:50.404 INFO: Change state on <bluesky.run_engine.RunEngine object at 0x7a235d478910> from 'running' -> 'idle'
2026-03-26 15:11:50.405 INFO: Cleaned up from plan <bluesky.utils.Plan object at 0x7a2310557820>


('97a56dad-f73f-4f11-8533-3b6c02d99422',
 'ba2e927e-ebcd-42b6-8be5-6ea3c2e9aa80',
 '0466ef5b-7669-428e-a999-94010d14ccfe',
 '3ee2c9c7-be60-4107-90aa-585e9a662b8e',
 '2e8e72d0-f277-4d83-8233-c89a9d2647a7')